# IMP CRS 24

In [41]:
from pathlib import Path


ROOT_DIR = Path("/mnt/data/Public/Public/colorectum/IMP-CRS2024-Dataset/CRS_Test")
SLIDE_DIR = ROOT_DIR / "slides"

In [42]:
import os

import mlflow


os.environ["MLFLOW_TRACKING_USERNAME"] = "..."
os.environ["MLFLOW_TRACKING_PASSWORD"] = "..."
mlflow.set_tracking_uri("https://mlflow.rationai.cloud.e-infra.cz/")

## Metadata

Table containing all the slide-level info

In [43]:
import pandas as pd


df = pd.read_csv(str(ROOT_DIR / "labels.csv"))

In [44]:
len(df)

900

In [45]:
df.columns

Index(['slide_name', 'slide_label'], dtype='object')

Slide-Level Labels

In [46]:
set(df["slide_label"].values)

{np.int64(0), np.int64(1), np.int64(2)}

All cases are classified within one of the three categories:

Non-neoplastic (label 0)

Low-grade lesions (label 1) - conventional adenomas with low-grade dysplasia

High-grade lesions (label 2) - conventional adenomas with high-grade dysplasia, intra-mucosal carcinomas and invasive adenocarcinomas.

In [47]:
label_to_descr = {0: "Non-Neoplastic", 1: "Low-grade lesions", 2: "High-grade lesions"}
df["slide_label_descr"] = df["slide_label"].map(label_to_descr)

In [48]:
df.head()

,slide_name,slide_label,slide_label_descr
0,CRC_test_000,0,Non-Neoplastic
1,CRC_test_001,1,Low-grade lesions
2,CRC_test_002,1,Low-grade lesions
3,CRC_test_003,1,Low-grade lesions
4,CRC_test_004,2,High-grade lesions


### Sumarize Counts

In [49]:
df["slide_label"].value_counts()

slide_label
1    454
2    264
0    182
Name: count, dtype: int64

### Export Enhanced Metadata Table

In [50]:
all_slides = list(SLIDE_DIR.rglob("*.svs"))

print(len(all_slides))

assert len(all_slides) == len(df)

900


Attach slide path to the table

In [51]:
stem_to_path = {p.stem: p for p in all_slides}
df["slide_path"] = df["slide_name"].map(stem_to_path)

In [52]:
df.head()

,slide_name,slide_label,slide_label_descr,slide_path
0,CRC_test_000,0,Non-Neoplastic,/mnt/data/Public/Public/colorectum/IMP-CRS2024...
1,CRC_test_001,1,Low-grade lesions,/mnt/data/Public/Public/colorectum/IMP-CRS2024...
2,CRC_test_002,1,Low-grade lesions,/mnt/data/Public/Public/colorectum/IMP-CRS2024...
3,CRC_test_003,1,Low-grade lesions,/mnt/data/Public/Public/colorectum/IMP-CRS2024...
4,CRC_test_004,2,High-grade lesions,/mnt/data/Public/Public/colorectum/IMP-CRS2024...


In [53]:
df.iloc[0]["slide_path"].exists()

True

In [56]:
df.to_csv("./imp_crs_24_test_full.csv", index=False)
mlflow.set_experiment("Colorectal cancer")

with mlflow.start_run(run_name="IMP CRS 24 Test Full Metadata"):
    mlflow.log_artifact("./imp_crs_24_test_full.csv")

🏃 View run IMP CRS 24 Test Full Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172/runs/b70ee1545dfc4e8caa506166a3ea83ef
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172


## Slides

In [31]:
import openslide


mpps = set()
level_counts = set()
downsamples = set()

for path in df["slide_path"]:
    slide = openslide.OpenSlide(path)
    mpp_x = slide.properties.get(openslide.PROPERTY_NAME_MPP_X)
    mpp_y = slide.properties.get(openslide.PROPERTY_NAME_MPP_Y)

    mpps.add((mpp_x, mpp_y))
    level_counts.add(slide.level_count)
    downsamples.add(slide.level_downsamples)

    slide.close()

In [32]:
mpps

{('0.262598', '0.262598'), ('0.26329999999999998', '0.26329999999999998')}

In [33]:
level_counts

{3, 4}

In [ ]:
downsamples

## Rescaled Slides

In [56]:
import openslide


loc = Path("/mnt/data/Public/Public/colorectum/IMP-CRS2024-Dataset/CRS_Test_rescaled")

mpps = set()
level_counts = set()
downsamples = set()
physical_sizes = set()

for path in loc.glob("*.tiff"):
    slide = openslide.OpenSlide(path)
    mpp_x = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_X))
    mpp_y = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_Y))

    mpps.add((mpp_x, mpp_y))
    level_counts.add(slide.level_count)
    downsamples.add(slide.level_downsamples)

    slide.close()

In [57]:
mpps

{(0.5251960227479631, 0.5251960227479631),
 (0.5265999845722661, 0.5265999845722661)}

In [58]:
level_counts

{7, 8, 9}

In [59]:
stem_to_rescaled_path = {file.stem: file for file in loc.glob("*.tiff")}
rescaled_df = pd.read_csv(
    mlflow.artifacts.download_artifacts(
        "mlflow-artifacts:/172/b70ee1545dfc4e8caa506166a3ea83ef/artifacts/imp_crs_24_test_full.csv"
    )
)
rescaled_df["stem"] = rescaled_df["slide_path"].map(lambda x: Path(x).stem)
rescaled_df["slide_path"] = rescaled_df["stem"].map(stem_to_rescaled_path)

In [60]:
rescaled_df

,slide_name,slide_label,slide_path,slide_label_descr,stem
0,CRC_test_000,0,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,Non-Neoplastic,CRC_test_000
1,CRC_test_001,1,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,Low-grade lesions,CRC_test_001
2,CRC_test_002,1,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,Low-grade lesions,CRC_test_002
3,CRC_test_003,1,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,Low-grade lesions,CRC_test_003
4,CRC_test_004,2,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,High-grade lesions,CRC_test_004
...,...,...,...,...,...
895,CRC_test_895,1,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,Low-grade lesions,CRC_test_895
896,CRC_test_896,2,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,High-grade lesions,CRC_test_896
897,CRC_test_897,0,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,Non-Neoplastic,CRC_test_897
898,CRC_test_898,0,/mnt/data/Public/Public/colorectum/IMP-CRS2024...,Non-Neoplastic,CRC_test_898


In [61]:
rescaled_df.to_csv("./imp_crs_24_test_rescaled.csv", index=False)
mlflow.set_experiment("Colorectal cancer")

with mlflow.start_run(run_name="IMP CRS 24 Test (Rescaled) Metadata"):
    mlflow.log_artifact("./imp_crs_24_test_rescaled.csv")

🏃 View run IMP CRS 24 Test (Rescaled) Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172/runs/e0adcc4ea838403cb2ba5457c83b21d4
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172


## Filter QC-marked slides

We ran our QC service on the rescaled slides, based on the results, we remove some of the slides:

In [62]:
import mlflow
import pandas as pd


qc_exclude = pd.read_csv(
    "/mnt/projects/colorectal_cancer/to_exclude_based_on_qc_imp_crs_test.csv"
)
rescaled_df = pd.read_csv(
    mlflow.artifacts.download_artifacts(
        "mlflow-artifacts:/172/e0adcc4ea838403cb2ba5457c83b21d4/artifacts/imp_crs_24_test_rescaled.csv"
    )
)

print(len(rescaled_df))
stems_to_exclude = set(qc_exclude["slide_stem"])
rescaled_df = rescaled_df[
    ~(rescaled_df["slide_path"].map(lambda x: Path(x).stem).isin(stems_to_exclude))
]
print(len(rescaled_df))

900
891


In [63]:
rescaled_df.to_csv("./imp_crs_24_test_rescaled_filtered.csv", index=False)
mlflow.set_experiment("Colorectal cancer")

with mlflow.start_run(run_name="IMP CRS Test (Rescaled & Filtered) Metadata"):
    mlflow.log_artifact("./imp_crs_24_test_rescaled_filtered.csv")

🏃 View run IMP CRS Test (Rescaled & Filtered) Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172/runs/be7f72bff2c24ea8af3d979397895ac0
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172


## Binarize Labels

Only high-grade lesions (label 2) are positive. Low-grade lesions (label 1) are adenomas with low-grade dysplasia (no carcinoma), so they are negative. We also export a variant without them (2 vs 0), analogous to BRACS "Non-Atypical".

Note that label 2 also contains adenomas with high-grade dysplasia and intra-mucosal carcinomas (similar to DCIS being positive in BRACS).

In [ ]:
import mlflow
import pandas as pd


filtered_df = pd.read_csv(
    mlflow.artifacts.download_artifacts(
        "mlflow-artifacts:/172/be7f72bff2c24ea8af3d979397895ac0/artifacts/imp_crs_24_test_rescaled_filtered.csv"
    )
)

# only high-grade lesions (label 2) are positive; low-grade lesions (label 1) are negative
filtered_df["carcinoma"] = filtered_df["slide_label"] == 2

# variant without the low-grade lesions (analogous to BRACS "Non-Atypical")
non_low_grade_df = filtered_df[filtered_df["slide_label"] != 1]

print(
    "with low-grade:",
    len(filtered_df),
    filtered_df["carcinoma"].value_counts().to_dict(),
)
print(
    "without low-grade:",
    len(non_low_grade_df),
    non_low_grade_df["carcinoma"].value_counts().to_dict(),
)

filtered_df.to_csv("./imp_crs_24_test_rescaled_filtered_binary.csv", index=False)
non_low_grade_df.to_csv(
    "./imp_crs_24_test_rescaled_filtered_binary_non_low_grade.csv", index=False
)
mlflow.set_experiment("Colorectal cancer")

with mlflow.start_run(
    run_name="IMP CRS Test (Rescaled & Filtered & Binarized) Metadata"
):
    mlflow.log_artifact("./imp_crs_24_test_rescaled_filtered_binary.csv")
    mlflow.log_artifact("./imp_crs_24_test_rescaled_filtered_binary_non_low_grade.csv")

with low-grade: 891 {False: 627, True: 264}
without low-grade: 444 {True: 264, False: 180}
🏃 View run IMP CRS Test (Rescaled & Filtered & Binarized) Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172/runs/cb48483a15324a619124b4a2ebd1a45a
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172
